# Занятие 10.10.2026

## Тема: управление циклом, `enumerate`, `zip` - углублённо

Три инструмента, которые уже встречались, но теперь разбираем их всерьёз:

1. `break` и `continue` - управление ходом цикла
2. `enumerate` - индекс вместе со значением
3. `zip` - несколько списков одновременно

Ячейки запускай по порядку и меняй значения - так понятнее.

---

# 1. `break` и `continue`

## 1.1. Напоминание

| Команда | Что делает | Куда переходит управление |
|---|---|---|
| `break` | Прерывает **весь** цикл | На первую строку после цикла |
| `continue` | Прерывает **текущий шаг** | На начало цикла, к следующему элементу |

Обе команды работают и в `for`, и в `while`. И обе относятся к **ближайшему** циклу,
внутри которого написаны.

In [ ]:
numbers = [3, 8, 12, 5, 20]

# break: нашли и вышли, остаток не проверяем
for n in numbers:
    print("смотрим", n)
    if n > 10:
        print("нашли:", n)
        break
print("после цикла")

print("---")

# continue: часть элементов пропускаем
for n in numbers:
    if n % 2 != 0:
        continue          # нечётные пропускаем
    print("чётное:", n)

## 1.2. `while True` + `break` - главный приём

Иногда условие выхода становится известно только **в середине** тела цикла:
сначала надо что-то получить, потом проверить. В такой ситуации в заголовок `while`
писать условие нечего - и тогда пишут `while True`, а выходят через `break`.

```python
while True:
    ...          # что-то делаем
    if пора_выходить:
        break
    ...          # продолжаем
```

`while True` - это «повторяй вечно». Без `break` внутри такой цикл никогда не кончится,
поэтому `break` здесь не роскошь, а обязательная часть конструкции.

Так устроены почти все программы, которые ждут ввода пользователя: заранее неизвестно,
когда он напишет «выход».

In [ ]:
# Имитация ввода: вместо реального input берём готовый список
user_inputs = ["привет", "как дела", "выход", "это не прочитают"]

i = 0
while True:
    command = user_inputs[i]
    i += 1

    if command == "выход":       # условие выхода проверяется в середине
        print("Пока!")
        break

    print("Ты написал:", command)

print("---")

# Тот же приём для накопления до порога
total = 0
step = 0
while True:
    step += 1
    total += step
    if total > 20:
        break
print(f"Сумма превысила 20 на шаге {step}, итог {total}")

## 1.3. `else` у цикла - «цикл прошёл без `break`»

У `for` и `while` может быть ветка `else`. Она выполняется, **если цикл закончился
естественно**, то есть `break` ни разу не сработал.

```python
for x in коллекция:
    if условие:
        break
else:
    print("не нашли")      # сюда попадём, только если break не случился
```

Это ровно тот случай, когда нужно различить «нашли» и «перебрали всё и не нашли».
Без `else` для этого приходится заводить отдельную переменную-флаг.

Читается непривычно: `else` тут означает не «иначе», а «если обошлось без `break`».

In [ ]:
products = ["хлеб", "молоко", "сыр"]

# Вариант с else
for product in products:
    if product == "масло":
        print("Масло есть")
        break
else:
    print("Масла нет в списке")       # break не случился - попали сюда

print("---")

# Тот же поиск, но успешный
for product in products:
    if product == "сыр":
        print("Сыр есть")
        break
else:
    print("Сыра нет")                 # не выполнится: был break

print("---")

# Как это выглядело бы без else - через флаг
found = False
for product in products:
    if product == "масло":
        found = True
        break
if not found:
    print("Масла нет (через флаг)")

## 1.4. Вложенные циклы: `break` выходит только из своего

Циклы можно вкладывать один в другой. Внешний цикл делает шаг - внутренний прокручивается
целиком; потом внешний делает следующий шаг, и так далее.

**Главное про `break` здесь:** он прерывает только тот цикл, внутри которого написан.
Внешний цикл продолжит работу как ни в чём не бывало.

Чтобы выйти сразу из обоих, есть два способа:

1. **Флаг** - переменная, которую проверяет внешний цикл (работает, но громоздко).
2. **Функция и `return`** - `return` выходит из функции целиком, а значит и из всех циклов.
   Этот способ обычно чище.

In [ ]:
# Вложенные циклы: таблица умножения 3x3
for i in range(1, 4):
    for j in range(1, 4):
        print(f"{i}*{j}={i*j}", end="  ")
    print()

print("---")

# break во внутреннем цикле: внешний продолжается
for i in range(1, 4):
    for j in range(1, 4):
        if j == 2:
            break              # выходим только из внутреннего
        print(f"i={i}, j={j}")
print("внешний цикл всё равно прошёл все три шага")

print("---")

# Как выйти из обоих: через функцию и return
def find_pair(limit):
    for i in range(1, 10):
        for j in range(1, 10):
            if i * j == limit:
                return i, j        # выходим сразу из обоих циклов
    return None

print(find_pair(12))

## 1.5. Ловушка: `continue` в `while`

В `for` переменная цикла меняется сама. В `while` её меняешь ты - вручную.
И если `continue` стоит **до** строки с изменением, эта строка никогда не выполнится.
Получится бесконечный цикл.

```python
i = 0
while i < 5:
    if i == 2:
        continue      # i навсегда останется равным 2
    print(i)
    i += 1            # до этой строки управление уже не доходит
```

**Правило:** в `while` увеличивай счётчик **до** любого `continue` - либо не используй
`continue` в `while` вовсе.

In [ ]:
# ПРАВИЛЬНО: счётчик увеличивается первым, continue ему не мешает
i = 0
while i < 5:
    i += 1                 # сначала шаг
    if i == 3:
        continue           # теперь пропуск безопасен
    print(i, end=" ")
print()

# Код ниже - тот самый бесконечный цикл. Запускать не надо,
# но посмотри и убедись, что понимаешь, почему он зависнет:
#
# i = 0
# while i < 5:
#     if i == 2:
#         continue
#     print(i)
#     i += 1

## 1.6. `break` или `return`?

Внутри функции часто можно обойтись без `break`: `return` и прерывает цикл,
и сразу отдаёт результат.

| Что нужно | Чем пользоваться |
|---|---|
| Нашёл и сразу отдаёшь наружу | `return` - короче и понятнее |
| Нашёл, но до выхода надо ещё что-то посчитать | `break`, потом код после цикла |
| Выйти сразу из нескольких вложенных циклов | `return` |

In [ ]:
# return - прямо из цикла
def first_over(numbers, limit):
    for n in numbers:
        if n > limit:
            return n          # и цикл прерван, и значение отдано
    return None               # цикл дошёл до конца - ничего не нашли

print(first_over([3, 8, 12, 20], 10))   # 12
print(first_over([1, 2], 10))           # None

print("---")

# break нужен, когда после цикла есть ещё работа
def count_before_stop(words):
    count = 0
    for word in words:
        if word == "стоп":
            break             # выходим, но посчитанное ещё пригодится
        count += 1
    return count

print(count_before_stop(["а", "б", "стоп", "в"]))   # 2

---

# 2. `enumerate`

## 2.1. Что он на самом деле отдаёт

`enumerate` на каждом шаге отдаёт **пару** из двух значений: номер и элемент.
Такая пара называется **кортеж** и записывается в круглых скобках: `(0, 'bmw')`.

Запись `for i, car in enumerate(cars)` - это **распаковка**: пара автоматически
раскладывается по двум переменным. Можно и не распаковывать - тогда в переменную
попадёт сама пара.

In [ ]:
cars = ["bmw", "audi", "kia"]

# С распаковкой - так пишут почти всегда
for i, car in enumerate(cars):
    print(i, car)

print("---")

# Без распаковки: видно, что это пары
for pair in enumerate(cars):
    print(pair, "| номер:", pair[0], "| значение:", pair[1])

print("---")

# Что внутри, если собрать всё сразу
print(list(enumerate(cars)))      # [(0, 'bmw'), (1, 'audi'), (2, 'kia')]

## 2.2. `start` меняет только нумерацию

`enumerate(cars, start=1)` начнёт считать с единицы. Это удобно для вывода человеку:
люди нумеруют списки с 1, а Python - с 0.

**Важно понимать:** `start` меняет только **выдаваемое число**. Сам список не сдвигается,
и если этим числом полезть в список по индексу, получишь не тот элемент или `IndexError`.
Номер для показа и индекс для доступа - разные вещи.

In [ ]:
cars = ["bmw", "audi", "kia"]

for number, car in enumerate(cars, start=1):
    print(f"{number}) {car}")

print("---")

# Ловушка: номер со start=1 не равен индексу
for number, car in enumerate(cars, start=1):
    print(f"номер {number}, элемент '{car}', а cars[{number}] - это", 
          cars[number] if number < len(cars) else "IndexError!")

## 2.3. Ловушка: `enumerate` со срезом

Если передать в `enumerate` срез, нумерация начнётся с нуля **от начала среза**,
а не от начала исходного списка. Срез - это уже новый список, он ничего не знает
о том, откуда его вырезали.

Чтобы получить настоящие индексы в исходном списке, нужно сдвинуть нумерацию
через `start`.

In [ ]:
letters = ["а", "б", "в", "г", "д"]

# Срез с третьего элемента
for i, letter in enumerate(letters[2:]):
    print(i, letter)          # 0 в, 1 г, 2 д - индексы НЕ от исходного списка

print("---")

# Чиним через start: сдвигаем нумерацию на размер отрезанной части
for i, letter in enumerate(letters[2:], start=2):
    print(i, letter, "| проверка:", letters[i])   # теперь индексы настоящие

## 2.4. Типовая задача: собрать индексы по условию

`enumerate` нужен ровно тогда, когда одного значения недостаточно - важна позиция.
У списков есть метод `.index(x)`, но он находит **только первое** вхождение
и не умеет искать по условию. `enumerate` умеет и то, и другое.

In [ ]:
marks = [5, 3, 5, 2, 5]

# Все позиции пятёрок - .index() так не умеет
positions = []
for i, mark in enumerate(marks):
    if mark == 5:
        positions.append(i)
print("пятёрки на позициях:", positions)

print("---")

# Поиск по условию с досрочным выходом
def first_bad_index(values):
    for i, value in enumerate(values):
        if value < 3:
            return i
    return -1

print(first_bad_index(marks))      # 3
print(first_bad_index([5, 4, 5]))  # -1

## 2.5. `enumerate` - не список

`enumerate(cars)` сам по себе ничего не считает и не хранит. Он выдаёт пары
**по одной, по мере надобности** - такие объекты называют ленивыми.

Поэтому `print(enumerate(cars))` покажет не пары, а что-то вроде
`<enumerate object at 0x...>`. Чтобы увидеть содержимое, оберни в `list(...)`.

Так же ведут себя `range` и `zip`. Плюс в том, что память не тратится: даже
для списка на миллион элементов `enumerate` не создаёт миллион пар заранее.

In [ ]:
cars = ["bmw", "audi"]

print(enumerate(cars))         # <enumerate object ...> - объект, а не данные
print(list(enumerate(cars)))   # [(0, 'bmw'), (1, 'audi')]
print(range(3))                # range(0, 3) - то же самое поведение
print(list(range(3)))          # [0, 1, 2]

---

# 3. `zip`

## 3.1. Напоминание и несколько списков

`zip` связывает элементы нескольких коллекций **по позиции**: первый с первым,
второй со вторым. На каждом шаге отдаёт кортеж, который удобно распаковать.

Списков может быть сколько угодно - просто перечисли их через запятую,
и переменных в цикле будет столько же.

In [ ]:
products = ["хлеб", "молоко", "сыр"]
prices = [50, 80, 300]
counts = [2, 1, 3]

for product, price in zip(products, prices):
    print(f"{product}: {price} руб.")

print("---")

# Три списка сразу
for product, price, count in zip(products, prices, counts):
    print(f"{product}: {count} шт. по {price} = {price * count} руб.")

print("---")

# Без распаковки видно, что это кортежи
print(list(zip(products, prices)))

## 3.2. Разная длина: `zip` молча обрезает

Если списки разной длины, `zip` остановится на **самом коротком**. Лишние элементы
просто не попадут в цикл - **без ошибки и без предупреждения**.

Это самая опасная черта `zip`: программа не падает, а молча считает не все данные.
Если списки должны быть одинаковой длины, проверь это сам.

In [ ]:
products = ["хлеб", "молоко", "сыр"]
prices = [50, 80]                      # цены на сыр забыли

print(list(zip(products, prices)))     # сыр потерялся, ошибки нет

total = 0
for product, price in zip(products, prices):
    total += price
print("Сумма посчитана по неполным данным:", total)

print("---")

# Подстраховка: сравнить длины заранее
if len(products) != len(prices):
    print(f"ВНИМАНИЕ: {len(products)} товаров, но {len(prices)} цен")

## 3.3. `zip` одноразовый

`zip`, как и `enumerate`, ленивый: он не хранит пары, а отдаёт их по одной.
И у этого есть неочевидное следствие - **пройти по одному и тому же `zip` дважды нельзя**.
После первого прохода он исчерпан, и второй цикл не выполнится ни разу.

Ошибка выглядит странно: первый цикл работает, второй «молчит», при этом никаких
сообщений нет.

**Как быть:** либо вызвать `zip(...)` второй раз, либо сохранить результат
в список через `list(zip(...))` - список можно перебирать сколько угодно.

In [ ]:
products = ["хлеб", "молоко"]
prices = [50, 80]

pairs = zip(products, prices)

print("первый проход:")
for product, price in pairs:
    print(" ", product, price)

print("второй проход:")
for product, price in pairs:
    print(" ", product, price)       # не выполнится ни разу - zip исчерпан
print("(пусто)")

print("---")

# Решение: сохранить в список
pairs = list(zip(products, prices))
for product, price in pairs:
    print("1-й раз:", product, price)
for product, price in pairs:
    print("2-й раз:", product, price)

## 3.4. Обратная операция: `zip(*pairs)`

`zip` умеет не только склеивать, но и **разбирать обратно**. Звёздочка перед
списком пар означает «передай элементы этого списка как отдельные аргументы»:
вместо одного списка пар `zip` получит каждую пару отдельным аргументом.

```python
pairs = [("хлеб", 50), ("молоко", 80)]
names, costs = zip(*pairs)      # ('хлеб', 'молоко'), (50, 80)
```

Результат - кортежи, а не списки. Если нужны именно списки, оберни в `list(...)`.

Тот же приём «переворачивает» таблицу: строки становятся столбцами.

In [ ]:
pairs = [("хлеб", 50), ("молоко", 80), ("сыр", 300)]

names, costs = zip(*pairs)
print(names)            # ('хлеб', 'молоко', 'сыр') - это кортеж
print(costs)
print(list(names))      # ['хлеб', 'молоко', 'сыр'] - если нужен список
print(sum(costs))       # 430 - с кортежем чисел работают те же функции

print("---")

# Таблица: строки <-> столбцы
table = [[1, 2, 3],
         [4, 5, 6]]
for column in zip(*table):
    print(column)       # (1, 4) / (2, 5) / (3, 6)

## 3.5. `zip` вместе с `enumerate`

Их можно сочетать: `enumerate(zip(a, b))` даёт номер и пару сразу.
Распаковка тогда двойная - обрати внимание на скобки вокруг пары.

```python
for i, (product, price) in enumerate(zip(products, prices), start=1):
```

Читается так: «номер `i`, а внутри пары - товар и цена».

Забегая вперёд: `dict(zip(ключи, значения))` собирает из двух списков **словарь** -
структуру «ключ: значение». Словари разберём на отдельном занятии, но приём
встречается так часто, что стоит его узнавать.

In [ ]:
products = ["хлеб", "молоко", "сыр"]
prices = [50, 80, 300]

for i, (product, price) in enumerate(zip(products, prices), start=1):
    print(f"{i}) {product} - {price} руб.")

print("---")

# Забегая вперёд: словарь из двух списков
menu = dict(zip(products, prices))
print(menu)                  # {'хлеб': 50, 'молоко': 80, 'сыр': 300}
print(menu["молоко"])        # 80 - доступ по названию, а не по индексу

---

# Частые ошибки по этим темам

| Ошибка | Что происходит | Как правильно |
|---|---|---|
| `break` вместо `continue` | Цикл обрывается на первом же неподходящем элементе | `continue` пропускает только текущий шаг |
| `continue` до `i += 1` в `while` | Бесконечный цикл, ячейка зависает | Увеличивать счётчик **до** `continue` |
| `while True` без `break` | Бесконечный цикл | `break` - обязательная часть этой конструкции |
| `break` во вложенном цикле | Выходит только из внутреннего | Флаг или функция с `return` |
| Номер из `enumerate(start=1)` как индекс | Не тот элемент или `IndexError` | Номер для показа, индекс для доступа - это разные числа |
| `enumerate` по срезу | Индексы считаются от начала среза | Сдвинуть через `start` |
| `zip` со списками разной длины | Часть данных молча теряется | Сравнить `len()` заранее |
| Второй проход по тому же `zip` | Цикл не выполняется, ошибки нет | `list(zip(...))` или вызвать `zip` снова |
| `print(zip(a, b))` вместо данных | Печатает `<zip object ...>` | `list(zip(a, b))` |

# Памятка

```python
break                  # выйти из цикла совсем
continue               # пропустить текущий шаг

while True:            # повторяй, пока не скажут стоп
    ...
    if условие:
        break

for x in lst:          # else срабатывает, если break НЕ случился
    if ...:
        break
else:
    print("не нашли")

for i, x in enumerate(lst):            # 0, 1, 2, ...
for i, x in enumerate(lst, start=1):   # 1, 2, 3, ...
for i, x in enumerate(lst[2:], start=2):   # настоящие индексы среза

for a, b in zip(lst1, lst2):           # по позициям, обрежет по короткому
for a, b, c in zip(l1, l2, l3):        # сколько угодно списков
pairs = list(zip(l1, l2))              # чтобы пройти несколько раз
names, costs = zip(*pairs)             # разобрать пары обратно
for i, (a, b) in enumerate(zip(l1, l2), start=1):   # номер + пара
```

**Ленивые объекты:** `range`, `enumerate`, `zip` не создают список заранее.
Печатать их напрямую бесполезно - оборачивай в `list(...)`.